# Ex3 · MNIST 분류기

In [ ]:
from __future__ import annotations

import torch

from torch import nn

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.uniform_(self.weight, -(in_features ** (-0.5)), in_features ** (-0.5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = x @ self.weight.transpose(-2, -1)
        return output if self.bias is None else output + self.bias

In [ ]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))
        nn.init.normal_(self.weight)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        return self.weight[idx]

In [ ]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        if not 0.0 <= p < 1.0:
            raise ValueError("p must be in [0, 1)")
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0.0:
            return x
        keep = (torch.rand_like(x) >= self.p).to(x.dtype)
        return x * keep / (1.0 - self.p)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-05, elementwise_affine: bool = True
    ):
        super().__init__()
        self.normalized_shape, self.eps = (normalized_shape, eps)
        self.weight = (
            nn.Parameter(torch.ones(normalized_shape)) if elementwise_affine else None
        )
        self.bias = (
            nn.Parameter(torch.zeros(normalized_shape)) if elementwise_affine else None
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, unbiased=False, keepdim=True)
        output = (x - mean) * torch.rsqrt(variance + self.eps)
        return output if self.weight is None else output * self.weight + self.bias

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-08):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (
            x
            * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            * self.weight
        )

In [ ]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        if depth < 1:
            raise ValueError("depth must be at least 1")
        dims = [in_dim] + [hidden_dim] * (depth - 1) + [out_dim]
        modules = []
        for index, (left, right) in enumerate(zip(dims[:-1], dims[1:], strict=True)):
            modules.append(Linear(left, right))
            if index < depth - 1:
                if use_layernorm:
                    modules.append(LayerNorm(right))
                modules.append(nn.GELU())
        self.net = nn.Sequential(*modules)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.in_proj, self.out_proj = (Linear(d_model, d_ff), Linear(d_ff, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out_proj(torch.nn.functional.gelu(self.in_proj(x)))

In [ ]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        return x + self.fn(x, *args, **kwargs)

In [ ]:
import os

from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms

In [ ]:
def cross_entropy_from_logits(
    logits: torch.Tensor, targets: torch.Tensor
) -> torch.Tensor:
    log_probs = torch.log_softmax(logits, dim=-1)
    return -log_probs.gather(1, targets.unsqueeze(1)).mean()

In [ ]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        self.projection = Linear(d_in, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.projection(x)

In [ ]:
def classifier_metrics(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    device = next(model.parameters()).device
    correct = total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for inputs, targets in loader:
            logits = model(inputs.to(device))
            targets = targets.to(device)
            correct += (logits.argmax(dim=-1) == targets).sum().item()
            loss_sum += (
                cross_entropy_from_logits(logits, targets).item() * targets.numel()
            )
            total += targets.numel()
    return (loss_sum / total, correct / total)


def accuracy(loader):
    return classifier_metrics(model, loader)[1]

In [ ]:
def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    losses, _ = train_classifier_with_history(
        model, train_data_loader, test_data_loader, lr, epochs, seed
    )
    return losses


def train_classifier_with_history(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int,
) -> tuple[list[float], list[tuple[float, float]]]:
    torch.manual_seed(seed)
    device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses, epoch_losses = ([], [])
    for epoch in range(epochs):
        model.train()
        train_loss_sum = train_examples = 0
        for inputs, targets in train_data_loader:
            inputs, targets = (inputs.to(device), targets.to(device))
            optimizer.zero_grad()
            loss = cross_entropy_from_logits(model(inputs), targets)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            train_loss_sum += losses[-1] * targets.numel()
            train_examples += targets.numel()
        train_loss = train_loss_sum / train_examples
        test_loss, test_accuracy = classifier_metrics(model, test_data_loader)
        epoch_losses.append((train_loss, test_loss))
        print(
            f"epoch {epoch + 1}/{epochs} | train loss {train_loss:.4f} | test loss {test_loss:.4f} | test accuracy {test_accuracy:.4f}"
        )
    return (losses, epoch_losses)

In [ ]:
training = datasets.MNIST(
    "data", train=True, transform=transforms.ToTensor(), download=True
)
testing = datasets.MNIST(
    "data", train=False, transform=transforms.ToTensor(), download=True
)
train_ds, test_ds = training, testing
smoke_mode = os.environ.get("HW_SMOKE") == "1"
requested_device = os.environ.get(
    "HW_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"
)
if requested_device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable")
device = torch.device(requested_device)
if smoke_mode:
    train_data = Subset(train_ds, range(2048))
    test_data = Subset(test_ds, range(512))
else:
    train_data, test_data = train_ds, test_ds
train_subset_loader = DataLoader(
    train_data, batch_size=128, shuffle=True, num_workers=0
)
test_subset_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=0)
torch.manual_seed(0)
model = nn.Sequential(
    nn.Flatten(),
    MLP(784, 256, 128, depth=2, use_layernorm=True),
    ClassificationHead(128, 10),
).to(device)
losses, epoch_losses = train_classifier_with_history(
    model,
    train_subset_loader,
    test_subset_loader,
    lr=1e-3,
    epochs=2 if smoke_mode else 5,
    seed=0,
)
final_accuracy = accuracy(test_subset_loader)
print(f"final accuracy: {final_accuracy:.4f}")
if smoke_mode and final_accuracy < 0.70:
    raise RuntimeError(f"MNIST smoke accuracy below 70%: {final_accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

train_curve, test_curve = zip(*epoch_losses)
plt.plot(range(1, len(train_curve) + 1), train_curve, label="Train")
plt.plot(range(1, len(test_curve) + 1), test_curve, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.legend()
plt.show()